# 04 · Model Training & Experiment Tracking

Trains a baseline (class-weighted logistic regression) and a gradient-boosted model
(XGBoost) over a small hyperparameter sweep, tracking every run's params/metrics/artifacts
in **MLflow** (SQLite-backed, so the Model Registry is fully usable locally). The best run
is registered as `fraud-xgboost` and promoted to the `staging` alias — the same registry
this project's real-time API and retraining pipeline (later phases) read from.

**Why class weighting instead of SMOTE/oversampling:** several features here are
time-windowed velocity aggregates (transactions in the trailing 1h/24h/7d). Interpolating
synthetic minority-class rows (as SMOTE does) would fabricate physically incoherent
combinations of those aggregates. `scale_pos_weight` reweights the loss instead of
touching the data, which is the safer default for temporally-structured fraud features.

**Why PR-AUC (average precision) as the primary model-selection metric:** with ~1.5-2%
positive rate, ROC-AUC is dominated by the (easy) true-negative volume and can look
deceptively strong. PR-AUC is far more sensitive to the trade-off that actually matters
operationally — precision at usable recall.

In [1]:
import os, json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.tracking import MlflowClient

with open("../data/processed/feature_columns.json") as f:
    cfg = json.load(f)
FEATURES, LABEL = cfg["features"], cfg["label"]

train = pd.read_parquet("../data/processed/train.parquet")
val = pd.read_parquet("../data/processed/val.parquet")
test = pd.read_parquet("../data/processed/test.parquet")

X_train, y_train = train[FEATURES], train[LABEL]
X_val, y_val = val[FEATURES], val[LABEL]
X_test, y_test = test[FEATURES], test[LABEL]

print(f"train={X_train.shape}  val={X_val.shape}  test={X_test.shape}")
print(f"train fraud rate={y_train.mean():.4%}  val={y_val.mean():.4%}  test={y_test.mean():.4%}")

train=(734002, 21)  val=(157287, 21)  test=(157286, 21)
train fraud rate=0.5925%  val=0.4476%  test=0.6059%


In [2]:
os.makedirs("../models", exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath('../mlflow.db')}")
mlflow.set_experiment("fraud-detection")
print("MLflow tracking URI:", mlflow.get_tracking_uri())

MLflow tracking URI: sqlite:////Users/crysis/fraud_detection/mlflow.db


## Baseline: class-weighted logistic regression

A simple, well-calibrated linear baseline. Any tree ensemble we ship needs to clearly beat this to justify its extra complexity and reduced interpretability.

In [3]:
with mlflow.start_run(run_name="logreg_baseline") as run:
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    logreg = LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0, random_state=42)
    logreg.fit(X_train_s, y_train)

    logreg_val_proba = logreg.predict_proba(X_val_s)[:, 1]
    logreg_val_roc_auc = roc_auc_score(y_val, logreg_val_proba)
    logreg_val_pr_auc = average_precision_score(y_val, logreg_val_proba)

    mlflow.log_params({"model": "logreg", "C": 1.0, "class_weight": "balanced"})
    mlflow.log_metrics({"val_roc_auc": logreg_val_roc_auc, "val_pr_auc": logreg_val_pr_auc})
    mlflow.sklearn.log_model(logreg, "model")

    logreg_run_id = run.info.run_id
    print(f"LogReg baseline  val ROC-AUC={logreg_val_roc_auc:.4f}  val PR-AUC={logreg_val_pr_auc:.4f}")

2026/08/31 10:31:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LogReg baseline  val ROC-AUC=0.9378  val PR-AUC=0.2102


## Second baseline: class-weighted Random Forest

A bagged tree ensemble sits between the linear baseline and the boosted-tree candidate —
worth checking whether XGBoost's sequential boosting actually earns its complexity over a
simpler, embarrassingly-parallel bagging approach on this feature set.

In [4]:
from sklearn.ensemble import RandomForestClassifier

# balanced_subsample (not plain "balanced") reweights each tree's own bootstrap sample --
# the correct variant for a bagging ensemble, matching this project's weighting-not-SMOTE
# philosophy above.
rf_param_grid = [
    dict(n_estimators=300, max_depth=10, min_samples_leaf=5),
    dict(n_estimators=300, max_depth=16, min_samples_leaf=5),
    dict(n_estimators=500, max_depth=None, min_samples_leaf=10),
]

rf_results = []
rf_models = {}

for params in rf_param_grid:
    run_name = f"rf_n{params['n_estimators']}_d{params['max_depth']}_leaf{params['min_samples_leaf']}"
    with mlflow.start_run(run_name=run_name) as run:
        rf = RandomForestClassifier(
            **params, class_weight="balanced_subsample", n_jobs=-1, random_state=42,
        )
        rf.fit(X_train, y_train)

        train_proba = rf.predict_proba(X_train)[:, 1]
        val_proba = rf.predict_proba(X_val)[:, 1]
        train_pr = average_precision_score(y_train, train_proba)
        roc = roc_auc_score(y_val, val_proba)
        pr = average_precision_score(y_val, val_proba)
        gap = train_pr - pr

        mlflow.log_params({**params, "class_weight": "balanced_subsample", "model": "random_forest"})
        mlflow.log_metrics({"val_roc_auc": roc, "val_pr_auc": pr, "train_pr_auc": train_pr,
                             "train_val_pr_gap": gap})
        mlflow.sklearn.log_model(rf, "model")

        rf_results.append({"run_id": run.info.run_id, "val_roc_auc": roc, "val_pr_auc": pr,
                            "train_pr_auc": train_pr, "gap": gap, **params})
        rf_models[run.info.run_id] = rf
        print(f"{run_name:40s}  val ROC-AUC={roc:.4f}  val PR-AUC={pr:.4f}  "
              f"train PR-AUC={train_pr:.4f}  gap={gap:.4f}")

rf_results_df = pd.DataFrame(rf_results).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
best_rf_pr_auc = rf_results_df.iloc[0].val_pr_auc
rf_results_df

2026/08/31 10:33:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


rf_n300_d10_leaf5                         val ROC-AUC=0.9830  val PR-AUC=0.5538  train PR-AUC=0.7315  gap=0.1777


2026/08/31 10:36:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


rf_n300_d16_leaf5                         val ROC-AUC=0.9870  val PR-AUC=0.6440  train PR-AUC=0.8977  gap=0.2537


2026/08/31 10:43:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


rf_n500_dNone_leaf10                      val ROC-AUC=0.9878  val PR-AUC=0.6890  train PR-AUC=0.9408  gap=0.2518


,run_id,val_roc_auc,val_pr_auc,train_pr_auc,gap,n_estimators,max_depth,min_samples_leaf
0,988c742972a245cebe6586dda84ef1e8,0.987786,0.689012,0.940792,0.251780,500,NaN,10
1,c85688028511455a93a473e9b844c341,0.986987,0.644034,0.897717,0.253683,300,16.0,5
2,ab3d7e6c671e48fcb8231ba5da6a89cc,0.982989,0.553752,0.731495,0.177742,300,10.0,5


## XGBoost hyperparameter sweep

A small, deliberately narrow grid (this is a portfolio project, not a Kaggle leaderboard
chase) — each configuration is logged as its own MLflow run for full comparability.

**Regularization added to the grid.** The original 4-config sweep only varied
`max_depth`/`learning_rate`/`n_estimators` — no `subsample`, `colsample_bytree`,
`min_child_weight`, or `reg_lambda`. Notebook 07's error analysis and the stacking
diagnostics in notebook 04e both found a sizeable train-vs-test PR-AUC gap (~0.30) for the
unregularized winner, consistent with `max_depth=8` memorizing training-set noise on a
rare-fraud class (~4,300 positive examples). Three additional configs test whether
subsampling rows/columns and penalizing leaf weights narrows that gap without giving up
test-set PR-AUC — cheap to check, no new data required.

In [5]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.1f}")

param_grid = [
    dict(max_depth=4, learning_rate=0.10, n_estimators=200),
    dict(max_depth=6, learning_rate=0.10, n_estimators=300),
    dict(max_depth=6, learning_rate=0.05, n_estimators=500),
    dict(max_depth=8, learning_rate=0.05, n_estimators=400),
    # Regularized variants of the strongest unregularized config -- row/column subsampling
    # plus min_child_weight and L2 leaf-weight penalties, to check whether they narrow the
    # train/test overfitting gap without costing val PR-AUC.
    dict(max_depth=8, learning_rate=0.05, n_estimators=400,
         subsample=0.8, colsample_bytree=0.8),
    dict(max_depth=8, learning_rate=0.05, n_estimators=400,
         min_child_weight=5, reg_lambda=5.0),
    dict(max_depth=6, learning_rate=0.05, n_estimators=500,
         subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=5.0),
]

sweep_results = []
sweep_models = {}

for params in param_grid:
    reg_bits = "_".join(f"{k}{v}" for k, v in params.items()
                         if k not in ("max_depth", "learning_rate", "n_estimators"))
    run_name = f"xgb_d{params['max_depth']}_lr{params['learning_rate']}_n{params['n_estimators']}"
    run_name += f"_{reg_bits}" if reg_bits else ""
    with mlflow.start_run(run_name=run_name) as run:
        model = xgb.XGBClassifier(
            **params,
            scale_pos_weight=scale_pos_weight,
            objective="binary:logistic",
            eval_metric="aucpr",
            random_state=42,
            n_jobs=-1,
            tree_method="hist",
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        train_proba = model.predict_proba(X_train)[:, 1]
        val_proba = model.predict_proba(X_val)[:, 1]
        train_pr = average_precision_score(y_train, train_proba)
        roc = roc_auc_score(y_val, val_proba)
        pr = average_precision_score(y_val, val_proba)
        gap = train_pr - pr

        mlflow.log_params({**params, "scale_pos_weight": round(scale_pos_weight, 2), "model": "xgboost"})
        mlflow.log_metrics({"val_roc_auc": roc, "val_pr_auc": pr, "train_pr_auc": train_pr,
                             "train_val_pr_gap": gap})
        mlflow.xgboost.log_model(model, "model")

        sweep_results.append({"run_id": run.info.run_id, "val_roc_auc": roc, "val_pr_auc": pr,
                               "train_pr_auc": train_pr, "gap": gap, **params})
        sweep_models[run.info.run_id] = model
        print(f"{run_name:55s}  val ROC-AUC={roc:.4f}  val PR-AUC={pr:.4f}  "
              f"train PR-AUC={train_pr:.4f}  gap={gap:.4f}")

results_df = pd.DataFrame(sweep_results).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
results_df

scale_pos_weight = 167.8


2026/08/31 10:44:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgb_d4_lr0.1_n200                                        val ROC-AUC=0.9928  val PR-AUC=0.7205  train PR-AUC=0.8107  gap=0.0902


2026/08/31 10:44:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgb_d6_lr0.1_n300                                        val ROC-AUC=0.9933  val PR-AUC=0.7339  train PR-AUC=0.9534  gap=0.2196


...

2026/08/31 10:45:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgb_d8_lr0.05_n400_min_child_weight5_reg_lambda5.0       val ROC-AUC=0.9932  val PR-AUC=0.7522  train PR-AUC=0.9748  gap=0.2225


2026/08/31 10:46:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgb_d6_lr0.05_n500_subsample0.8_colsample_bytree0.8_min_child_weight5_reg_lambda5.0  val ROC-AUC=0.9936  val PR-AUC=0.7463  train PR-AUC=0.9317  gap=0.1854


,run_id,val_roc_auc,val_pr_auc,train_pr_auc,gap,max_depth,learning_rate,n_estimators,subsample,colsample_bytree,min_child_weight,reg_lambda
0,b68e524cdf5e4384b5575ebb5f5577eb,0.993221,0.752236,0.974785,0.222549,8,0.05,400,NaN,NaN,5.0,5.0
1,e0e7e8fe43a74fcaa460e2fb4dedb6f8,0.992313,0.752005,0.983892,0.231887,8,0.05,400,0.8,0.8,NaN,NaN
2,c3521156eda14fab972ac22c74013c40,0.992151,0.746348,0.984185,0.237837,8,0.05,400,NaN,NaN,NaN,NaN
3,3daad8efa96f432e82003f5c15996cff,0.993650,0.746285,0.931702,0.185417,6,0.05,500,0.8,0.8,5.0,5.0
4,5842003ba4d2492cb0cac5ed420680c3,0.993308,0.733865,0.953444,0.219579,6,0.10,300,NaN,NaN,NaN,NaN
5,09a1d9b6e01a4bf8b752a439a5a7d811,0.993152,0.726878,0.938833,0.211955,6,0.05,500,NaN,NaN,NaN,NaN
6,014723c402b4416da83156b676c9a52e,0.992807,0.720485,0.810720,0.090235,4,0.10,200,NaN,NaN,NaN,NaN


## Select and register the best model

Promoted to the `staging` alias in the MLflow Model Registry — this is the handoff point where `05_model_evaluation_explainability.ipynb` picks the model up for a rigorous hold-out evaluation before anything is promoted further.

In [6]:
best = results_df.iloc[0]
best_run_id = best.run_id
best_model = sweep_models[best_run_id]

print(f"Best config: depth={int(best.max_depth)} lr={best.learning_rate} n_estimators={int(best.n_estimators)}")
print(f"XGBoost val PR-AUC={best.val_pr_auc:.4f}  vs  "
      f"Random Forest best val PR-AUC={best_rf_pr_auc:.4f}  vs  "
      f"LogReg baseline val PR-AUC={logreg_val_pr_auc:.4f}")
if best_rf_pr_auc > best.val_pr_auc:
    print("NOTE: Random Forest beat the XGBoost sweep winner -- would need to reconsider "
          "the deployed model, not just log it as a baseline.")
else:
    print("XGBoost's sequential boosting still wins on this feature set -- Random Forest "
          "stays a rejected-but-evaluated alternative, same role as the LogReg baseline.")

model_uri = f"runs:/{best_run_id}/model"
registered = mlflow.register_model(model_uri, "fraud-xgboost")

client = MlflowClient()
client.set_registered_model_alias("fraud-xgboost", "staging", registered.version)
print(f"Registered 'fraud-xgboost' version {registered.version} -> alias 'staging'")

Registered model 'fraud-xgboost' already exists. Creating a new version of this model...
2026/08/31 10:46:24 WARNING mlflow.tracking._model_registry.fluent: Run with id b68e524cdf5e4384b5575ebb5f5577eb has no artifacts at artifact path 'model', registering model based on models:/m-70939dc267114eaa907d725f756d626e instead


Best config: depth=8 lr=0.05 n_estimators=400
XGBoost val PR-AUC=0.7522  vs  Random Forest best val PR-AUC=0.6890  vs  LogReg baseline val PR-AUC=0.2102
XGBoost's sequential boosting still wins on this feature set -- Random Forest stays a rejected-but-evaluated alternative, same role as the LogReg baseline.
Registered 'fraud-xgboost' version 14 -> alias 'staging'


Created version '14' of model 'fraud-xgboost'.


## Persist a portable model artifact

Saved outside MLflow too (`models/`) so the FastAPI scoring service can load it directly without depending on a running MLflow tracking server in production.

In [7]:
best_model.save_model("../models/fraud_xgboost.json")
with open("../models/feature_columns.json", "w") as f:
    json.dump({"features": FEATURES, "label": LABEL, "mlflow_run_id": best_run_id,
               "mlflow_model_version": registered.version}, f, indent=2)

print("Saved models/fraud_xgboost.json + models/feature_columns.json")
print("\nRun `mlflow ui --backend-store-uri sqlite:///mlflow.db` from the project root to browse experiments.")

Saved models/fraud_xgboost.json + models/feature_columns.json

Run `mlflow ui --backend-store-uri sqlite:///mlflow.db` from the project root to browse experiments.
